In [ ]:
# PMST S2 — 48h month-tail test-only（本 notebook 仅此一格可运行）
# 滑窗 reshape 须与 PMST_s2_data.py / monthtail 一致：transpose(0, 1, 3, 2)，与 repeat(win_end, ns) 标签对齐。
# 滑窗必须为 transpose(0,1,3,2)，与 PMST_s2_data / monthtail 训练集一致；曾误用 (0,2,1,3) 会导致特征与能见度标签错位。

# 自包含：无需先跑其他单元格。
# 优化点：只生成 month-tail test；在 build 阶段先按 visibility 有效性过滤，再立刻按月末 test 过滤。

import os
import re
import gc
import warnings
from functools import lru_cache

import numpy as np
import pandas as pd
import xarray as xr
import pvlib
from tqdm import tqdm
from numpy.lib.stride_tricks import sliding_window_view

warnings.filterwarnings("ignore")

from PMST_s2_data import (
    VAR_MAPPING,
    FINAL_FEATURE_ORDER,
    TRAIN_RATIO,
    VAL_RATIO,
    GAP_HOURS,
    WINDOW_SIZE,
    STEP_SIZE,
    UNIQUE_VEG_IDS,
    MAX_VIS_THRESHOLD,
    compute_fog_features,
    calculate_dewpoint_from_rh,
    calculate_zenith_angle,
    get_nearest_veg,
    extract_terrain,
)

# ================= 配置 =================
BASE_PATH = "/public/home/putianshu/vis_mlp"
CURRENT_48H_DIR = os.path.join(BASE_PATH, "tianji_auto_station", "current_48h")
VIS_SOURCE_NC = os.path.join(BASE_PATH, "tianji_auto_station", "merged_final_all_vars.nc")
VEG_FILE = "/public/home/putianshu/vis_cnn/data_vegtype.nc"
ORO_FILE = "/public/home/putianshu/vis_cnn/data_orography.nc"

OUTPUT_DATASET_DIR_TESTONLY = os.path.join(BASE_PATH, "ml_dataset_fe_12h_48h_pm10_testonly_leadtime")

# pm10: S2
PM10_S2_FILE = os.path.join(BASE_PATH, "pm10_station", "pm10_station_s2_2025.nc")

# current_48h 中变量列表（2m 温度为 TMP2m）
VARIABLES_48H = [
    "rh2m",
    "PRATEsfc",
    "slp",
    "DSWRFsfc",
    "UGRD10m",
    "VGRD10m",
    "cape",
    "cldl",
    "t925",
    "rh925",
    "u925",
    "v925",
    "dp1000",
    "dp925",
    "q1000",
    "q925",
    "omg925",
    "omg1000",
    "gust",
    "TMP2m",
]

# month-tail v2
VAL_LAST_DAYS = 3
TEST_LAST_DAYS = 3
GAP_HOURS_SPLIT = 24

# 与 PMST_net_test_11_s2_pm10 / PMST_s2_data_48h_pm10.py 对齐
EXPECTED_DYN_VARS = 26
# compute_fog_features 现为 32 维 + 周期 4 维 = 36；lead_hour 仅在 meta，不进 FE（与训练 checkpoint 一致）
EXPECTED_FE_DIM = 36

# 仅处理与 12h 测试集 reference 时刻存在重叠的起报，避免扫全年 ~730 个 run
REF_META_12H_TEST = os.path.join(BASE_PATH, "ml_dataset_s2_tianji_12h_pm10_monthtail_2", "meta_test.csv")

# 调试时可限制起报数，例如 MAX_RUNS = 20
MAX_RUNS = None


def filter_runs_overlapping_12h_test(run_list, meta_12h_path):
    """保留满足 ∃t∈ref_test: init≤t≤init+48h 的起报（与 48h 预报有效时段重叠）。"""
    if not os.path.exists(meta_12h_path):
        print(f"[WARN] Ref 12h meta not found: {meta_12h_path}; using all {len(run_list)} runs.", flush=True)
        return run_list
    m12 = pd.read_csv(meta_12h_path, parse_dates=["time"])
    test_times = pd.DatetimeIndex(m12["time"].unique()).sort_values()
    ts = np.asarray(test_times.astype("datetime64[ns]")).astype(np.int64)
    test_min_ns = int(ts.min())
    test_max_ns = int(ts.max())
    kept = []
    for run_str in run_list:
        init = pd.to_datetime(run_str, format="%Y%m%d%H")
        t_end = init + pd.Timedelta(hours=48)
        init_ns = int(init.value)
        end_ns = int(t_end.value)
        if end_ns < test_min_ns or init_ns > test_max_ns:
            continue
        i = np.searchsorted(ts, init_ns, side="left")
        j = np.searchsorted(ts, end_ns, side="right")
        if j > i:
            kept.append(run_str)
    print(f"Filtered runs (overlap 12h test times): {len(kept)} / {len(run_list)}", flush=True)
    return kept


def get_run_list_from_current_48h():
    pattern = os.path.join(CURRENT_48H_DIR, f"{VARIABLES_48H[0]}_*_0-48h_IDW.nc")
    import glob

    files = glob.glob(pattern)
    runs = []
    for f in files:
        base = os.path.basename(f)
        m = re.match(r"^[^_]+_(\d{10})_0-48h_IDW\.nc$", base)
        if m:
            runs.append(m.group(1))
    return sorted(set(runs))


@lru_cache(maxsize=1)
def _load_station_latlon():
    station_df = pd.read_csv(os.path.join(BASE_PATH, "tianji_auto_station", "station_info.csv"))
    station_df = station_df[
        (station_df["station_lon"] >= 65)
        & (station_df["station_lon"] <= 145)
        & (station_df["station_lat"] >= 10)
        & (station_df["station_lat"] <= 60)
    ]
    return station_df.set_index("num_station")


def load_merged_run_ds(run_str, data_veg, data_oro):
    """为某个起报 run_str 拼接 current_48h 变量，并计算 D2M/DPD/INVERSION。"""
    init_time = pd.to_datetime(run_str, format="%Y%m%d%H")

    ds_list = []
    for var in VARIABLES_48H:
        path = os.path.join(CURRENT_48H_DIR, f"{var}_{run_str}_0-48h_IDW.nc")
        if not os.path.exists(path):
            return None, None
        ds_list.append(xr.open_dataset(path))

    ds = xr.merge(ds_list, join="inner")
    for d in ds_list:
        d.close()
    del ds_list
    gc.collect()

    if "num_station" in ds.dims:
        ds = ds.rename({"num_station": "station_id"})

    station_idx = _load_station_latlon()
    sids = ds.station_id.values

    sids_idx = pd.Index(np.asarray(sids))
    valid_sids = sids_idx.intersection(station_idx.index)
    if len(valid_sids) == 0:
        return None, None

    if len(valid_sids) < len(sids_idx):
        ds = ds.sel(station_id=valid_sids.values)
        sids = ds.station_id.values

    lats = station_idx.loc[sids]["station_lat"].values
    lons = station_idx.loc[sids]["station_lon"].values
    ds = ds.assign_coords(lat=("station_id", lats), lon=("station_id", lons))

    times = pd.to_datetime(ds.time.values)
    lead_sec = (times - init_time).total_seconds()
    lead_hours = (lead_sec / 3600.0).astype(np.float32)
    ds["lead_time"] = (("time",), lead_hours)
    ds = ds.sortby("time")

    rename_map = {k: v for k, v in VAR_MAPPING.items() if k in ds}
    ds = ds.rename(rename_map)

    if "t2mz" in ds and "T2M" not in ds:
        ds = ds.rename({"t2mz": "T2M"})

    if "D2M" not in ds and "T2M" in ds and "RH2M" in ds:
        ds["D2M"] = calculate_dewpoint_from_rh(ds["T2M"], ds["RH2M"])
    if "DPD" not in ds and "T2M" in ds and "D2M" in ds:
        ds["DPD"] = ds["T2M"] - ds["D2M"]
    if "INVERSION" not in ds and "T_925" in ds and "T2M" in ds:
        ds["INVERSION"] = ds["T_925"] - ds["T2M"]

    # 常用派生：风速/风向（用于 compute_fog_features 内的 summer/regime 诊断）
    if "WSPD10" not in ds and "U10" in ds and "V10" in ds:
        ds["WSPD10"] = np.sqrt(ds["U10"] ** 2 + ds["V10"] ** 2)
    if "WDIR10" not in ds and "U10" in ds and "V10" in ds:
        ds["WDIR10"] = np.degrees(np.arctan2(-ds["U10"], -ds["V10"])) % 360
    if "WSPD925" not in ds and "U_925" in ds and "V_925" in ds:
        ds["WSPD925"] = np.sqrt(ds["U_925"] ** 2 + ds["V_925"] ** 2)

    return ds, init_time


def get_monthly_split_mask_last_days(sample_times, gap_hours, val_last_days, test_last_days):
    """与 s2_data_monthtail_v2.ipynb 一致：Month-tail + gap 以整日计。"""
    times = pd.DatetimeIndex(sample_times)
    month_periods = times.to_period("M")
    n = len(times)
    train_mask = np.zeros(n, dtype=bool)
    val_mask = np.zeros(n, dtype=bool)
    test_mask = np.zeros(n, dtype=bool)

    if gap_hours % 24 != 0:
        raise ValueError(f"gap_hours must be a multiple of 24 for day-based split, got {gap_hours}")
    gap_days = gap_hours // 24

    for month_period in month_periods.unique():
        start = month_period.start_time
        end = month_period.end_time
        month_mask = month_periods == month_period
        dim = month_period.days_in_month

        d_test0 = dim - test_last_days + 1
        d_val1 = d_test0 - gap_days - 1
        d_val0 = d_val1 - val_last_days + 1
        d_train_end = d_val0 - gap_days - 1

        if d_train_end < 1 or d_val0 < 1 or d_val1 < d_val0:
            print(
                f"  [WARN] split skip month {month_period}: d_train_end={d_train_end}, d_val0={d_val0}, d_val1={d_val1}, d_test0={d_test0}",
                flush=True,
            )
            continue

        t_train_end = pd.Timestamp(month_period.year, month_period.month, d_train_end) + pd.Timedelta(
            hours=23, minutes=59, seconds=59
        )
        t_val0 = pd.Timestamp(month_period.year, month_period.month, d_val0)
        t_val1 = pd.Timestamp(month_period.year, month_period.month, d_val1) + pd.Timedelta(
            hours=23, minutes=59, seconds=59
        )
        t_test0 = pd.Timestamp(month_period.year, month_period.month, d_test0)

        train_mask |= month_mask & (times >= start) & (times <= t_train_end)
        val_mask |= month_mask & (times >= t_val0) & (times <= t_val1)
        test_mask |= month_mask & (times >= t_test0) & (times <= end)

    return train_mask, val_mask, test_mask


def build_windows_and_features_testonly(ds_run, init_time, run_str, data_veg, data_oro, vis_da, pm10_da):
    """只返回 month-tail test 的样本；模型输入与当前 S2 checkpoint 保持同构。run_str=起报 YYYYMMDDHH。"""
    ds = ds_run
    lats = ds["lat"].values
    lons = ds["lon"].values
    times = pd.to_datetime(ds.time.values)
    stations = ds.station_id.values
    nt, ns = len(times), len(stations)

    X_met = ds[FINAL_FEATURE_ORDER].to_array(dim="v").transpose("time", "station_id", "v").values.astype(np.float32)
    zenith = calculate_zenith_angle(lats, lons, times)
    X_dyn = np.concatenate([X_met, zenith], axis=-1)
    del X_met, zenith
    gc.collect()

    if pm10_da is None:
        pm10_grid = np.zeros((nt, ns), dtype=np.float32)
    else:
        time_vals = pm10_da["time"].values
        if np.issubdtype(time_vals.dtype, np.datetime64):
            time_index = pd.DatetimeIndex(time_vals)
        else:
            time_index = pd.to_datetime(time_vals, unit="s", origin="unix")
        sid_index = pd.Index(pm10_da["station_id"].values)
        station_cast = stations.astype(pm10_da["station_id"].dtype)
        time_pos = time_index.get_indexer(times, method="nearest")
        sid_pos = sid_index.get_indexer(station_cast)

        pm10_grid = np.full((nt, ns), np.nan, dtype=np.float32)
        base = np.asarray(pm10_da.transpose("time", "station_id").values).reshape(-1)
        ns_pm10 = len(sid_index)
        linear_idx = time_pos[:, None] * ns_pm10 + sid_pos[None, :]
        ok_mask = (time_pos[:, None] >= 0) & (sid_pos[None, :] >= 0)
        if ok_mask.any():
            pm10_grid[ok_mask] = base[linear_idx[ok_mask]].astype(np.float32)

        pm10_grid = np.maximum(pm10_grid, 0.0)
        pm10_grid = pm10_grid * 1e12
        pm10_median = np.nanmedian(pm10_grid)
        if not np.isfinite(pm10_median):
            pm10_median = 0.0
        pm10_grid = np.where(np.isfinite(pm10_grid), pm10_grid, pm10_median).astype(np.float32)

    X_dyn = np.concatenate([X_dyn, pm10_grid[..., None]], axis=-1)
    nv = X_dyn.shape[-1]
    if nv != EXPECTED_DYN_VARS:
        raise RuntimeError(f"dyn vars mismatch: got {nv}, expected {EXPECTED_DYN_VARS}")

    n_wins = (nt - WINDOW_SIZE) // STEP_SIZE + 1
    if n_wins <= 0:
        return None

    X_wins = sliding_window_view(X_dyn, WINDOW_SIZE, axis=0)[::STEP_SIZE]
    # 与 PMST_s2_data.py / monthtail 一致
    X_wins = X_wins.transpose(0, 1, 3, 2).reshape(-1, WINDOW_SIZE, nv)

    win_end_idx = (np.arange(n_wins) * STEP_SIZE + (WINDOW_SIZE - 1)).astype(int)
    win_end_times = times[win_end_idx]
    win_lead = ds["lead_time"].values[win_end_idx].astype(np.float32)

    m_t = np.repeat(win_end_times, ns)
    m_s = np.tile(stations, n_wins)
    m_la = np.tile(lats, n_wins)
    m_lo = np.tile(lons, n_wins)
    m_lead = np.repeat(win_lead, ns)

    if vis_da is None:
        y_flat = np.full(n_wins * ns, np.nan, dtype=np.float32)
    else:
        vis_sel = vis_da.sel(time=win_end_times, method="nearest")
        vis_sel = vis_sel.reindex(station_id=stations)
        y_flat = vis_sel.values.astype(np.float32).ravel()

    valid_mask = (~np.isnan(y_flat)) & (y_flat >= 0) & (y_flat <= MAX_VIS_THRESHOLD)
    if valid_mask.sum() == 0:
        return None

    y_flat = y_flat[valid_mask]
    X_wins = X_wins[valid_mask]
    m_t = m_t[valid_mask]
    m_s = m_s[valid_mask]
    m_la = m_la[valid_mask]
    m_lo = m_lo[valid_mask]
    m_lead = m_lead[valid_mask]

    valid_times = pd.DatetimeIndex(m_t)
    _, _, test_m = get_monthly_split_mask_last_days(valid_times, GAP_HOURS_SPLIT, VAL_LAST_DAYS, TEST_LAST_DAYS)
    test_m = np.asarray(test_m, dtype=bool)
    if test_m.sum() == 0:
        return None

    y_flat = y_flat[test_m]
    X_wins = X_wins[test_m]
    m_t = m_t[test_m]
    m_s = m_s[test_m]
    m_la = m_la[test_m]
    m_lo = m_lo[test_m]
    m_lead = m_lead[test_m]

    veg_raw = get_nearest_veg(lats, lons, data_veg)
    type_to_idx = {v: i for i, v in enumerate(UNIQUE_VEG_IDS)}
    feat_veg = np.array([type_to_idx.get(v, 0) for v in veg_raw])[:, None].astype(np.float32)
    feat_oro = extract_terrain(lats, lons, data_oro)
    X_stat = np.concatenate(
        [
            (lats[:, None] / 90.0).astype(np.float32),
            (lons[:, None] / 180.0).astype(np.float32),
            feat_oro,
            feat_veg,
        ],
        axis=1,
    )

    station_pos = {sid: i for i, sid in enumerate(stations)}
    sample_station_idx = np.array([station_pos.get(sid, -1) for sid in m_s], dtype=int)
    if (sample_station_idx < 0).any():
        keep = sample_station_idx >= 0
        y_flat = y_flat[keep]
        X_wins = X_wins[keep]
        m_t = m_t[keep]
        m_s = m_s[keep]
        m_la = m_la[keep]
        m_lo = m_lo[keep]
        m_lead = m_lead[keep]
        sample_station_idx = sample_station_idx[keep]

    X_stat_flat = X_stat[sample_station_idx].astype(np.float32)
    fe_fog = compute_fog_features(X_wins, WINDOW_SIZE, nv)

    sample_times = pd.DatetimeIndex(m_t)
    months = sample_times.month.values.astype(np.float32)
    hours = sample_times.hour.values.astype(np.float32)
    time_feat = np.column_stack(
        [
            np.sin(2 * np.pi * months / 12),
            np.cos(2 * np.pi * months / 12),
            np.sin(2 * np.pi * hours / 24),
            np.cos(2 * np.pi * hours / 24),
        ]
    ).astype(np.float32)

    fe_flat = np.concatenate([fe_fog, time_feat], axis=1).astype(np.float32)
    if fe_flat.shape[1] != EXPECTED_FE_DIM:
        raise RuntimeError(f"FE dim mismatch: got {fe_flat.shape[1]}, expected {EXPECTED_FE_DIM}")

    X_dyn_flat = X_wins.reshape(X_wins.shape[0], -1).astype(np.float32)
    m_init = np.full(len(m_t), run_str, dtype=object)

    return X_dyn_flat, X_stat_flat, fe_flat, y_flat.astype(np.float32), (m_t, m_s, m_la, m_lo, m_lead, m_init)


def save_testonly_from_arrays(X_dyn_flat, X_stat_flat, fe_flat, y_flat, meta, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    times_all, stats_all, lats_all, lons_all, lead_all, init_all = meta

    N = len(y_flat)
    assert X_dyn_flat.shape[0] == N
    assert X_stat_flat.shape[0] == N
    assert fe_flat.shape[0] == N

    np.save(os.path.join(out_dir, "y_test.npy"), y_flat)
    pd.DataFrame(
        {
            "time": times_all,
            "station_id": stats_all,
            "lat": lats_all,
            "lon": lons_all,
            "lead_hour": lead_all.astype(np.float32),
            "init_time": init_all,
        }
    ).to_csv(os.path.join(out_dir, "meta_test.csv"), index=False)

    dims = [X_dyn_flat.shape[1], X_stat_flat.shape[1], fe_flat.shape[1]]
    fp = np.lib.format.open_memmap(
        os.path.join(out_dir, "X_test.npy"),
        mode="w+",
        dtype="float32",
        shape=(N, sum(dims)),
    )

    bs = 100000
    for i in tqdm(range(0, N, bs), desc="write", total=(N + bs - 1) // bs):
        bi = slice(i, min(i + bs, N))
        fp[bi, : dims[0]] = X_dyn_flat[bi]
        fp[bi, dims[0] : dims[0] + dims[1]] = X_stat_flat[bi]
        fp[bi, dims[0] + dims[1] :] = fe_flat[bi]

    del fp
    gc.collect()


def main_testonly_monthtail():
    print("Loading auxiliary data...", flush=True)
    data_veg = xr.open_dataset(VEG_FILE, engine="h5netcdf")
    data_oro = xr.open_dataset(ORO_FILE, engine="h5netcdf")

    if not os.path.exists(VIS_SOURCE_NC):
        print(f"Visibility source not found: {VIS_SOURCE_NC}", flush=True)
        vis_da = None
    else:
        ds_vis = xr.open_dataset(VIS_SOURCE_NC, engine="h5netcdf")
        if "vis" in ds_vis:
            ds_vis = ds_vis.rename({"vis": "visibility"})
        ds_vis = ds_vis.assign_coords(time=ds_vis.time - pd.Timedelta(hours=8))
        vis_da = ds_vis["visibility"]

    if os.path.exists(PM10_S2_FILE):
        pm10_ds = xr.open_dataset(PM10_S2_FILE, engine="h5netcdf")
        pm10_da = pm10_ds["pm10"]
    else:
        print(f"[WARN] pm10 file not found: {PM10_S2_FILE}", flush=True)
        pm10_da = None

    run_list = get_run_list_from_current_48h()
    run_list = filter_runs_overlapping_12h_test(run_list, REF_META_12H_TEST)
    if MAX_RUNS is not None:
        run_list = run_list[: int(MAX_RUNS)]
        print(f"MAX_RUNS={MAX_RUNS}: using first {len(run_list)} runs", flush=True)
    print(f"Found {len(run_list)} runs (after test-calendar filter). Generating month-tail test-only...", flush=True)

    X_dyn_list, X_stat_list, fe_list, y_list = [], [], [], []
    meta_t_list, meta_s_list, meta_la_list, meta_lo_list, meta_lead_list, meta_init_list = [], [], [], [], [], []

    for run_str in tqdm(run_list, desc="Runs"):
        ds_run, init_time = load_merged_run_ds(run_str, data_veg, data_oro)
        if ds_run is None:
            continue
        try:
            out = build_windows_and_features_testonly(
                ds_run, init_time, run_str, data_veg, data_oro, vis_da, pm10_da
            )
        except Exception as e:
            print(f"  Run {run_str} skip: {e}", flush=True)
            out = None
        finally:
            ds_run.close()
            gc.collect()

        if out is None:
            continue

        X_dyn_flat, X_stat_flat, fe_flat, y_flat, meta = out
        X_dyn_list.append(X_dyn_flat)
        X_stat_list.append(X_stat_flat)
        fe_list.append(fe_flat)
        y_list.append(y_flat)
        meta_t_list.append(meta[0])
        meta_s_list.append(meta[1])
        meta_la_list.append(meta[2])
        meta_lo_list.append(meta[3])
        meta_lead_list.append(meta[4])
        meta_init_list.append(meta[5])

    if not X_dyn_list:
        print("No test samples produced.", flush=True)
        return

    X_dyn_flat = np.concatenate(X_dyn_list, axis=0)
    X_stat_flat = np.concatenate(X_stat_list, axis=0)
    fe_flat = np.concatenate(fe_list, axis=0)
    y_flat = np.concatenate(y_list, axis=0)
    m_t = np.concatenate(meta_t_list, axis=0)
    m_s = np.concatenate(meta_s_list, axis=0)
    m_la = np.concatenate(meta_la_list, axis=0)
    m_lo = np.concatenate(meta_lo_list, axis=0)
    m_lead = np.concatenate(meta_lead_list, axis=0)
    m_init = np.concatenate(meta_init_list, axis=0)

    print(f"Final shapes: X_dyn={X_dyn_flat.shape}, X_stat={X_stat_flat.shape}, FE={fe_flat.shape}", flush=True)
    print(f"Expected total dim = {WINDOW_SIZE * EXPECTED_DYN_VARS + 5 + 1 + EXPECTED_FE_DIM}", flush=True)

    save_testonly_from_arrays(
        X_dyn_flat,
        X_stat_flat,
        fe_flat,
        y_flat,
        (m_t, m_s, m_la, m_lo, m_lead, m_init),
        OUTPUT_DATASET_DIR_TESTONLY,
    )

    print(f"Done. Output: {OUTPUT_DATASET_DIR_TESTONLY}", flush=True)


main_testonly_monthtail()


Loading auxiliary data...
Filtered runs (overlap 12h test times): 120 / 730
Found 120 runs (after test-calendar filter). Generating month-tail test-only...


Runs: 100%|██████████| 120/120 [05:35<00:00,  2.80s/it]


Final shapes: X_dyn=(5336668, 312), X_stat=(5336668, 6), FE=(5336668, 36)
Expected total dim = 354


write: 100%|██████████| 54/54 [00:24<00:00,  2.23it/s]


Done. Output: /public/home/putianshu/vis_mlp/ml_dataset_fe_12h_48h_pm10_testonly_leadtime


...

  Run 2025123012 skip: FE dim mismatch: got 37, expected 36


Runs: 100%|█████████▉| 728/730 [34:01<00:06,  3.20s/it]

  Run 2025123100 skip: FE dim mismatch: got 37, expected 36


Runs: 100%|██████████| 730/730 [34:08<00:00,  2.81s/it]

No test samples produced.
